# 14 Strategy | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: if-else zamiast algorytmow (naruszenie OCP)
2. 🎯 Struktura: Context i Strategy
3. 🐍 Strategia jako funkcja (`sorted(key=)`)
4. 📋 Rejestr strategii i strategia parametryzowana
5. 🛒 Zastosowania: sortowanie, walidacja, platnosci

## 1. 🔹 Problem: if-else zamiast algorytmow

Strategy (Strategia) to wzorzec behawioralny pozwalajacy
zdefiniowac rodzine algorytmow i umiescic je w osobnych
klasach, oraz umozliwic ich zamienne uzywanie.

Problem: wielka if-else zamist algorytmow
```python
def sort(data, method):
    if method == 'bubble': ...
    elif method == 'quick': ...
    elif method == 'merge': ...
    # Dodanie nowego: zmiana tej funkcji!
```

Naruszenia SOLID:
- OCP: dodanie nowej strategii wymaga zmiany istniejacego kodu
- SRP: jedna klasa zawiera wiele algorytmow
- Trudne testowanie: musimy testowac caly if-else razem

Strategy rozwiazuje:
- Kazdy algorytm w osobnej klasie
- Context deleguje do Strategy przez interfejs
- Nowe algorytmy: nowa klasa, bez zmiany Context

> 💡 Gdy widzisz if-else gdzie warunek nie zmienia
> struktury ale algorytm - pomysl o Strategy.

In [ ]:
# Problem: if-else narusza OCP
def calculate_discount(price: float, customer_type: str) -> float:
    if customer_type == 'regular':
        return price * 0.05
    elif customer_type == 'premium':
        return price * 0.15
    elif customer_type == 'vip':
        return price * 0.25
    # Dodanie 'employee' -> zmiana tutaj!
    else:
        return 0.0

# Problem: testujemy cala funkcje, nie pojedynczy algorytm
print('Regular:', calculate_discount(100, 'regular'))
print('Premium:', calculate_discount(100, 'premium'))
print('VIP:', calculate_discount(100, 'vip'))

# Jak naprawic? Wyodrebnic algorytmy do osobnych jednostek
print('\nProblemy:')
for p in [
    'Dodanie employee wymaga zmiany calculate_discount',
    'Testowanie: musimy pokryc caly if-else',
    'Trudno zmienic algorytm w runtime bez przekazywania stringa',
    'Logika rabatow jest ukryta w jednej funkcji',
]:
    print(f'- {p}')

---

### 🐍 Cwiczenia - problem OCP

1. Policz ile miejsc w kodzie musisz zmienic gdy dodasz
   'student' (10% rabatu) do `calculate_discount`.
2. Napisz `compress(data, method)` z if-else dla `zip`, `gzip`,
   `bz2`. Ile klas testowych potrzebujesz?
3. *(Trudniejsze)* Napisz wersje z if-else i wersje ze Strategy
   dla kalkulatora stawek podatkowych. Porownaj testowalnosc.

In [ ]:
# Cwiczenie 1: koszt zmiany
def analyze_change_cost():
    places_to_change = [
        'calculate_discount - dodac elif',
        'testy calculate_discount - nowy przypadek testowy',
    ]
    print(f'Miejsc do zmiany: {len(places_to_change)}')
    for p in places_to_change:
        print(f'  - {p}')
    print('Z Strategy: 1 miejsce (nowa klasa StudentDiscount)')

analyze_change_cost()

In [ ]:
# Cwiczenie 2: compress z if-else
import zlib, gzip, bz2

def compress(data: bytes, method: str) -> bytes:
    if method == 'zip':
        return zlib.compress(data)
    elif method == 'gzip':
        return gzip.compress(data)
    elif method == 'bz2':
        return bz2.compress(data)
    else:
        raise ValueError(f'Unknown: {method}')

data = b'test data ' * 50
for m in ['zip', 'gzip', 'bz2']:
    compressed = compress(data, m)
    ratio = len(compressed) / len(data)
    print(f'{m}: {ratio:.2f} ratio')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: podatki if-else vs Strategy
def calculate_tax_if_else(income: float, country: str) -> float:
    if country == 'PL':
        if income < 120000: return income * 0.12
        return 14400 + (income - 120000) * 0.32
    elif country == 'DE':
        return income * 0.19
    elif country == 'UK':
        return income * 0.20
    return 0.0

print('if-else:')
for country in ['PL', 'DE', 'UK']:
    print(f'  {country}: {calculate_tax_if_else(100000, country):.2f}')

print('Testowalnosc if-else: musimy testowac cala funkcje')
print('Z Strategy: testujemy kazda strategie niezaleznie')

## 2. 🔹 Struktura: Context i Strategy

Uczestnicy wzorca Strategy:

**Strategy** (interfejs):
- Definiuje wspolny interfejs dla wszystkich algorytmow
- Zazwyczaj jedna metoda: `execute()`, `sort()`, `compress()`

**ConcreteStrategy**:
- Implementuje konkretny algorytm
- Nie zna Context - pracuje tylko na przekazanych danych

**Context**:
- Trzyma referencje do Strategy
- Deleguje prace do strategii
- Moze zmienic strategie w runtime (`set_strategy()`)
- Moze przekazywac do strategii swoj stan lub dane

Relacja Context-Strategy:
- Context zleca strategie wywolanie (`call through`)
- Strategy nie zna Context (luzsne sprzezenie)
- Dane przekazywane przez parametry lub closure

In [ ]:
from abc import ABC, abstractmethod

# Strategy interface
class DiscountStrategy(ABC):
    @abstractmethod
    def calculate(self, price: float) -> float: ...
    @abstractmethod
    def description(self) -> str: ...

# ConcreteStrategies
class NoDiscount(DiscountStrategy):
    def calculate(self, price: float) -> float: return 0.0
    def description(self) -> str: return 'No discount'

class PercentageDiscount(DiscountStrategy):
    def __init__(self, percent: float): self._percent = percent
    def calculate(self, price: float) -> float: return price * (self._percent / 100)
    def description(self) -> str: return f'{self._percent}% discount'

class FixedDiscount(DiscountStrategy):
    def __init__(self, amount: float): self._amount = amount
    def calculate(self, price: float) -> float: return min(self._amount, price)
    def description(self) -> str: return f'{self._amount} PLN discount'

class BuyTwoGetOneDiscount(DiscountStrategy):
    def __init__(self, item_price: float): self._item_price = item_price
    def calculate(self, price: float) -> float:
        items = int(price / self._item_price)
        free = items // 3
        return free * self._item_price
    def description(self) -> str: return 'Buy 2 get 1 free'

# Context
class ShoppingCart:
    def __init__(self, strategy: DiscountStrategy = None):
        self._items: list[dict] = []
        self._strategy = strategy or NoDiscount()

    def add_item(self, name: str, price: float) -> None:
        self._items.append({'name': name, 'price': price})

    def set_discount(self, strategy: DiscountStrategy) -> None:
        self._strategy = strategy

    def checkout(self) -> dict:
        total = sum(i['price'] for i in self._items)
        discount = self._strategy.calculate(total)
        return {
            'items': len(self._items),
            'total': total,
            'discount': discount,
            'discount_desc': self._strategy.description(),
            'final': total - discount,
        }

# Uzywamy roznych strategii dla tego samego koszyka
cart = ShoppingCart()
cart.add_item('Widget', 50.0)
cart.add_item('Gadget', 100.0)
cart.add_item('Thing', 30.0)

strategies = [
    NoDiscount(),
    PercentageDiscount(10),
    PercentageDiscount(20),
    FixedDiscount(25),
    BuyTwoGetOneDiscount(50.0),
]

for strategy in strategies:
    cart.set_discount(strategy)
    result = cart.checkout()
    print(f'{result["discount_desc"]:25}: total={result["total"]}, '
          f'discount={result["discount"]:.2f}, final={result["final"]:.2f}')

---

### 🐍 Cwiczenia - Context / Strategy

1. Dodaj `SeasonalDiscount(season: str)` - 5% zima, 10% lato.
   Przetestuj bez zmiany `ShoppingCart`.
2. Napisz `DataSorter(strategy)` z interfejsem `SortStrategy.sort(data)`.
   Implementacje: `BubbleSort`, `QuickSort`, `PythonSort`.
3. *(Trudniejsze)* Napisz `CompressionContext(strategy)` ktora
   mierzy czas i wspoliczynnik kompresji dla kazdej strategii.

In [ ]:
# Cwiczenie 1: SeasonalDiscount
class SeasonalDiscount(DiscountStrategy):
    RATES = {'spring': 0.05, 'summer': 0.10, 'autumn': 0.07, 'winter': 0.05}
    def __init__(self, season: str):
        self._season = season
    def calculate(self, price: float) -> float:
        return price * self.RATES.get(self._season, 0)
    def description(self) -> str:
        rate = self.RATES.get(self._season, 0)
        return f'{self._season} discount ({rate*100:.0f}%)'

cart.set_discount(SeasonalDiscount('summer'))
print(cart.checkout())

In [ ]:
# Cwiczenie 2: DataSorter
class SortStrategy(ABC):
    @abstractmethod
    def sort(self, data: list) -> list: ...

class BubbleSort(SortStrategy):
    def sort(self, data: list) -> list:
        arr = data[:]
        for i in range(len(arr)):
            for j in range(len(arr) - i - 1):
                if arr[j] > arr[j+1]: arr[j], arr[j+1] = arr[j+1], arr[j]
        return arr

class QuickSort(SortStrategy):
    def sort(self, data: list) -> list:
        if len(data) <= 1: return data[:]
        pivot = data[len(data) // 2]
        return (self.sort([x for x in data if x < pivot]) +
                [x for x in data if x == pivot] +
                self.sort([x for x in data if x > pivot]))

class PythonSort(SortStrategy):
    def sort(self, data: list) -> list: return sorted(data)

class DataSorter:
    def __init__(self, strategy: SortStrategy): self._strategy = strategy
    def set_strategy(self, s: SortStrategy) -> None: self._strategy = s
    def sort(self, data: list) -> list: return self._strategy.sort(data)

data = [64, 34, 25, 12, 22, 11, 90]
sorter = DataSorter(BubbleSort())
print('Bubble:', sorter.sort(data))
sorter.set_strategy(QuickSort())
print('Quick:', sorter.sort(data))
sorter.set_strategy(PythonSort())
print('Python:', sorter.sort(data))

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: CompressionContext
import zlib, gzip, time

class CompressionStrategy2(ABC):
    @abstractmethod
    def compress(self, data: bytes) -> bytes: ...
    @abstractmethod
    def name(self) -> str: ...

class ZlibStrategy(CompressionStrategy2):
    def compress(self, data: bytes) -> bytes: return zlib.compress(data)
    def name(self) -> str: return 'zlib'

class GzipStrategy(CompressionStrategy2):
    def compress(self, data: bytes) -> bytes: return gzip.compress(data)
    def name(self) -> str: return 'gzip'

class CompressionContext:
    def __init__(self, strategy: CompressionStrategy2):
        self._strategy = strategy
    def set_strategy(self, s: CompressionStrategy2) -> None: self._strategy = s
    def compress(self, data: bytes) -> dict:
        start = time.perf_counter()
        result = self._strategy.compress(data)
        elapsed = time.perf_counter() - start
        return {
            'strategy': self._strategy.name(),
            'original': len(data),
            'compressed': len(result),
            'ratio': len(result) / len(data),
            'time_ms': elapsed * 1000,
        }

ctx = CompressionContext(ZlibStrategy())
big_data = b'repeated pattern ' * 10000
for strat in [ZlibStrategy(), GzipStrategy()]:
    ctx.set_strategy(strat)
    r = ctx.compress(big_data)
    print(f'{r["strategy"]}: ratio={r["ratio"]:.3f}, time={r["time_ms"]:.2f}ms')

## 3. 🔹 Strategia jako funkcja (`sorted(key=`)

W Python funkcje sa pierwszorzednymi obywatelami - mozna
uzywac ich jako strategii bez tworzenia osobnych klas.

`sorted(key=func)` to Strategy w akcji:
- `key=lambda x: x['age']` - strategia sortowania po wieku
- `key=str.lower` - strategia sortowania case-insensitive
- `key=len` - strategia sortowania po dlugosci

Zalety funkcji jako strategii:
- Krocej - nie trzeba klasy dla prostych przypadkow
- `lambda`, `functools.partial`, zwykle funkcje

Kiedy uzyc klasy a kiedy funkcji:
- **Funkcja**: prosta logika, jednorazowa, bez stanu
- **Klasa**: zlozona logika, parametryzowana, z metodami
  pomocniczymi, powtarzana w wielu miejscach

`functools.partial` jako fabryka strategii:
```python
sort_by_age = partial(sorted, key=lambda x: x['age'])
```

In [ ]:
from typing import Callable
import functools

# sorted() jako klasyczny przyklad Strategy
people = [
    {'name': 'Charlie', 'age': 30, 'salary': 5000},
    {'name': 'Alice', 'age': 25, 'salary': 7000},
    {'name': 'Bob', 'age': 35, 'salary': 4500},
]

print('By name:', [p['name'] for p in sorted(people, key=lambda x: x['name'])])
print('By age:', [p['name'] for p in sorted(people, key=lambda x: x['age'])])
print('By salary desc:', [p['name'] for p in sorted(people, key=lambda x: x['salary'], reverse=True)])

# Klasa Context z funkcja jako strategia
class DataProcessor:
    def __init__(self, transform: Callable = None, filter_fn: Callable = None):
        self._transform = transform or (lambda x: x)
        self._filter = filter_fn or (lambda x: True)

    def process(self, data: list) -> list:
        return [self._transform(x) for x in data if self._filter(x)]

# Rozne strategie jako lambda/funkcja
numbers = list(range(1, 11))

print('All doubled:', DataProcessor(transform=lambda x: x * 2).process(numbers))
print('Evens only:', DataProcessor(filter_fn=lambda x: x % 2 == 0).process(numbers))
print('Evens squared:', DataProcessor(
    transform=lambda x: x**2,
    filter_fn=lambda x: x % 2 == 0
).process(numbers))

# functools.partial jako fabryka strategii
from functools import partial

def multiply(x: float, factor: float) -> float: return x * factor
double = partial(multiply, factor=2)
triple = partial(multiply, factor=3)

for fn_name, fn in [('double', double), ('triple', triple)]:
    result = DataProcessor(transform=fn).process([1, 2, 3, 4, 5])
    print(f'{fn_name}: {result}')

---

### 🐍 Cwiczenia - funkcja jako strategia

1. Uzyj `sorted(key=)` z trzema roznymi kluczami na liscie
   slownikow z polami `name`, `price`, `rating`.
2. Napisz `Pipeline(steps: list[Callable])` przetwarzajacy
   dane przez lancuch transformacji (compose functions).
3. *(Trudniejsze)* Napisz `Validator` ktory przyjmuje
   liste `(name, Callable)` i zwraca slownik bledow.

In [ ]:
# Cwiczenie 1: sorted z kluczami
products = [
    {'name': 'Widget', 'price': 9.99, 'rating': 4.5},
    {'name': 'Gadget', 'price': 29.99, 'rating': 3.8},
    {'name': 'Doohickey', 'price': 4.99, 'rating': 4.9},
    {'name': 'Thingamajig', 'price': 19.99, 'rating': 4.1},
]

print('By name:', [p['name'] for p in sorted(products, key=lambda p: p['name'])])
print('By price:', [p['name'] for p in sorted(products, key=lambda p: p['price'])])
print('By rating desc:', [p['name'] for p in sorted(products, key=lambda p: p['rating'], reverse=True)])

In [ ]:
# Cwiczenie 2: Pipeline
class Pipeline:
    def __init__(self, *steps: Callable):
        self._steps = steps
    def process(self, data):
        result = data
        for step in self._steps:
            result = step(result)
        return result

# Pipeline dla tekstu
text_pipeline = Pipeline(
    str.strip,
    str.lower,
    lambda s: s.replace(' ', '_'),
)
print(text_pipeline.process('  Hello World  '))

# Pipeline dla liczb
num_pipeline = Pipeline(
    lambda lst: [x for x in lst if x > 0],  # tylko dodatnie
    lambda lst: [x * 2 for x in lst],         # podwoj
    lambda lst: sorted(lst, reverse=True),    # sortuj
)
print(num_pipeline.process([-1, 3, -2, 5, 1, 4]))

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Validator z mapowaniem bledow
class FuncValidator:
    def __init__(self):
        self._rules: list[tuple[str, Callable]] = []

    def add_rule(self, name: str, rule: Callable) -> 'FuncValidator':
        self._rules.append((name, rule))
        return self

    def validate(self, value) -> dict:
        # hint: zwroc {rule_name: bool} dla kazdej reguly
        return {name: rule(value) for name, rule in self._rules}

    def is_valid(self, value) -> bool:
        return all(self.validate(value).values())

password_rules = (FuncValidator()
    .add_rule('not_empty', bool)
    .add_rule('min_8_chars', lambda s: len(s) >= 8)
    .add_rule('has_digit', lambda s: any(c.isdigit() for c in s))
    .add_rule('has_upper', lambda s: any(c.isupper() for c in s)))

for pwd in ['', 'short', 'longbutnoupper1', 'LongGood1']:
    results = password_rules.validate(pwd)
    failures = [k for k, v in results.items() if not v]
    print(f'{pwd!r:20}: {"OK" if not failures else failures}')

## 4. 🔹 Rejestr strategii i strategia parametryzowana

Rejestr strategii (Strategy Registry):
- Slownik mapujacy klucz (string) na strategie
- Mozna laczyc z dekoratorem `@register`
- Umozliwia tworzenie strategii po nazwie

```python
STRATEGIES = {
    'card': CardPayment,
    'blik': BlikPayment,
}
payment = STRATEGIES[method]()
```

Strategia parametryzowana (Parameterized Strategy):
- Strategia ktora przyjmuje parametry w __init__
- Pozwala dostosowac zachowanie bez tworzenia wielu klas
- `PercentageDiscount(10)`, `PercentageDiscount(20)` - ta sama klasa

Alternatywa: closure jako fabryka strategii:
```python
def discount_strategy(percent):
    def strategy(price): return price * percent / 100
    return strategy

strategy10 = discount_strategy(10)
strategy20 = discount_strategy(20)
```

In [ ]:
from abc import ABC, abstractmethod
from typing import Callable

# Rejestr z dekoratorem
class StrategyRegistry:
    def __init__(self) -> None:
        self._strategies: dict[str, Callable] = {}

    def register(self, name: str) -> Callable:
        def decorator(cls_or_func):
            self._strategies[name] = cls_or_func
            return cls_or_func
        return decorator

    def create(self, name: str, *args, **kwargs):
        if name not in self._strategies:
            raise KeyError(f'Strategy {name!r} not registered')
        creator = self._strategies[name]
        return creator(*args, **kwargs) if callable(creator) else creator

    def list_strategies(self) -> list[str]:
        return list(self._strategies.keys())

payment_registry = StrategyRegistry()

@payment_registry.register('card')
def card_payment(amount: float) -> bool:
    print(f'Card: {amount} PLN -> processing...')
    return True

@payment_registry.register('blik')
def blik_payment(amount: float) -> bool:
    print(f'BLIK: {amount} PLN -> processing...')
    return True

@payment_registry.register('transfer')
def transfer_payment(amount: float) -> bool:
    print(f'Transfer: {amount} PLN -> scheduled')
    return True

print('Available:', payment_registry.list_strategies())
for method in ['card', 'blik', 'transfer']:
    strategy = payment_registry.create(method)
    strategy(99.99)


# Strategia parametryzowana jako closure
def make_discount(percent: float) -> Callable:
    """Fabryka strategii rabatowych."""
    def strategy(price: float) -> float:
        return price * percent / 100
    strategy.__name__ = f'discount_{percent}pct'
    return strategy

discounts = [
    ('regular', make_discount(5)),
    ('premium', make_discount(15)),
    ('vip', make_discount(25)),
]
for name, strategy in discounts:
    print(f'{name}: discount = {strategy(100):.1f} PLN')

---

### 🐍 Cwiczenia - rejestr i parametryzacja

1. Rozszerz `payment_registry` o `paypal` i `cash`.
   Napisz `PaymentProcessor` ktory korzysta z rejestru.
2. Napisz `make_validator(min_len, max_len, pattern)` -
   fabryka strategii walidacji zwracajaca lambda.
3. *(Trudniejsze)* Napisz `StrategyChain` laczacy wiele
   strategii (jak pipeline) - wynik jednej idzie do nastepnej.

In [ ]:
# Cwiczenie 1: rozszerzenie rejestru
@payment_registry.register('paypal')
def paypal_payment(amount: float) -> bool:
    print(f'PayPal: {amount} PLN')
    return True

@payment_registry.register('cash')
def cash_payment(amount: float) -> bool:
    print(f'Cash: {amount} PLN (handle carefully)')
    return True

class PaymentProcessor:
    def __init__(self, registry: StrategyRegistry):
        self._registry = registry
    def pay(self, method: str, amount: float) -> bool:
        strategy = self._registry.create(method)
        return strategy(amount)

processor = PaymentProcessor(payment_registry)
print('All methods:', payment_registry.list_strategies())
processor.pay('paypal', 49.99)
processor.pay('cash', 10.00)

In [ ]:
# Cwiczenie 2: make_validator
import re

def make_validator(min_len: int = 0, max_len: int = 999, pattern: str = None) -> Callable:
    def validator(value: str) -> tuple[bool, str]:
        if len(value) < min_len:
            return False, f'min {min_len} chars'
        if len(value) > max_len:
            return False, f'max {max_len} chars'
        if pattern and not re.match(pattern, value):
            return False, f'must match {pattern}'
        return True, ''
    return validator

username_validator = make_validator(min_len=3, max_len=20, pattern=r'^[a-zA-Z0-9_]+$')
for username in ['ab', 'valid_user', 'this_is_too_long_username', 'bad name!']:
    valid, msg = username_validator(username)
    print(f'{username!r:25}: {"OK" if valid else msg}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: StrategyChain
class StrategyChain:
    def __init__(self, *strategies: Callable):
        self._strategies = strategies

    def execute(self, data):
        result = data
        for strategy in self._strategies:
            result = strategy(result)
        return result

# Pipeline transformacji tekstu
text_chain = StrategyChain(
    str.strip,
    str.lower,
    lambda s: ' '.join(s.split()),  # normalizuj spacje
    lambda s: s.replace(' ', '-'),   # slug
)

texts = ['  Hello World  ', 'PYTHON Design Patterns', '  multiple   spaces  ']
for text in texts:
    print(f'{text!r} -> {text_chain.execute(text)!r}')

## 5. 🔹 Zastosowania: sortowanie, walidacja, platnosci

Strategy jest jednym z najczesciej uzywanych wzorcow:

**Sortowanie**:
- `sorted(data, key=strategy)` - Python built-in
- Rozne kryteria sortowania bez zmiany kodu sortowania

**Walidacja**:
- Zestaw regul jako liste strategii
- Latwe dodawanie/usuwanie regul
- Reuzowalnosc regul miedzy formularzami

**Platnosci**:
- Rozne metody platnosci jako strategie
- Dodanie nowej metody = nowa klasa
- Context (koszyk) nie zmienia sie

**Kompresja**: zlib, gzip, bz2 jako strategie

**Haszowanie**: SHA256, MD5 jako strategie

**Routowanie**: najkrotsza, najszybsza, eko droga

Zwiazek z innymi wzorcami:
- **Template Method**: dziedziczy szkielet, strategia composuje
- **Command**: enkapsuluje akcje, strategia enkapsuluje algorytm
- **State**: zmiana zachowania przez zmiane stanu
- **Decorator**: dodaje zachowanie, strategia zastepuje algorytm

In [ ]:
# Kompletny przyklad: platnosci z wieloma strategiami
from abc import ABC, abstractmethod
from dataclasses import dataclass
import time

@dataclass
class PaymentResult:
    success: bool
    transaction_id: str
    amount: float
    method: str
    message: str = ''

class PaymentStrategy(ABC):
    @abstractmethod
    def pay(self, amount: float, **details) -> PaymentResult: ...
    @abstractmethod
    def validate(self, **details) -> bool: ...
    @property
    @abstractmethod
    def name(self) -> str: ...

class CardPayment(PaymentStrategy):
    @property
    def name(self) -> str: return 'credit_card'
    def validate(self, **details) -> bool:
        return bool(details.get('card_number'))
    def pay(self, amount: float, **details) -> PaymentResult:
        card = details.get('card_number', '')[-4:]
        return PaymentResult(True, f'CARD-{int(time.time())}', amount, self.name, f'**** {card}')

class BlikPayment(PaymentStrategy):
    @property
    def name(self) -> str: return 'blik'
    def validate(self, **details) -> bool:
        code = details.get('code', '')
        return len(code) == 6 and code.isdigit()
    def pay(self, amount: float, **details) -> PaymentResult:
        return PaymentResult(True, f'BLIK-{int(time.time())}', amount, self.name)

class PayPalPayment(PaymentStrategy):
    @property
    def name(self) -> str: return 'paypal'
    def validate(self, **details) -> bool:
        return '@' in details.get('email', '')
    def pay(self, amount: float, **details) -> PaymentResult:
        return PaymentResult(True, f'PP-{int(time.time())}', amount, self.name, details['email'])

class Checkout:
    def __init__(self, strategy: PaymentStrategy):
        self._strategy = strategy

    def set_payment_method(self, strategy: PaymentStrategy) -> None:
        self._strategy = strategy

    def process_payment(self, amount: float, **details) -> PaymentResult:
        if not self._strategy.validate(**details):
            return PaymentResult(False, '', amount, self._strategy.name, 'Validation failed')
        return self._strategy.pay(amount, **details)

checkout = Checkout(CardPayment())

results = [
    checkout.process_payment(99.99, card_number='4111111111111234'),
    (checkout.set_payment_method(BlikPayment()),
     checkout.process_payment(49.50, code='123456'))[1],
    (checkout.set_payment_method(PayPalPayment()),
     checkout.process_payment(199.00, email='alice@paypal.com'))[1],
    checkout.process_payment(50.00, email='invalid'),  # walidacja nie przechodzi
]
for r in results:
    status = 'OK' if r.success else 'FAILED'
    print(f'{status} | {r.method:15} | {r.amount:7.2f} PLN | {r.transaction_id or r.message}')

---

### 🐍 Cwiczenia - zastosowania

1. Napisz `SearchStrategy` dla wyszukiwarki produktow:
   `ExactMatch`, `FuzzyMatch`, `RegexMatch`. Context: `ProductSearch`.
2. Napisz `TaxStrategy` z implementacjami dla PL, DE, UK
   korzystajac z rejestru strategii.
3. *(Trudniejsze)* Napisz `BackoffStrategy` dla ponownych prob:
   `ConstantBackoff(delay)`, `ExponentialBackoff(base)`,
   `LinearBackoff(increment)`. Przetestuj z retry loop.

In [ ]:
# Cwiczenie 1: SearchStrategy
class SearchStrategy(ABC):
    @abstractmethod
    def matches(self, query: str, text: str) -> bool: ...

class ExactMatch(SearchStrategy):
    def matches(self, query: str, text: str) -> bool:
        return query.lower() in text.lower()

class FuzzyMatch(SearchStrategy):
    def __init__(self, threshold: float = 0.6): self.threshold = threshold
    def matches(self, query: str, text: str) -> bool:
        # prosta heurystyka: ile slow zapytania w tekscie
        words = set(query.lower().split())
        text_words = set(text.lower().split())
        return len(words & text_words) / len(words) >= self.threshold if words else False

import re
class RegexMatch(SearchStrategy):
    def matches(self, query: str, text: str) -> bool:
        try: return bool(re.search(query, text, re.IGNORECASE))
        except re.error: return False

class ProductSearch:
    def __init__(self, strategy: SearchStrategy): self._strategy = strategy
    def search(self, query: str, products: list[dict]) -> list[dict]:
        return [p for p in products if self._strategy.matches(query, p['name'])]

products = [
    {'name': 'Wireless Mouse', 'price': 49},
    {'name': 'Mechanical Keyboard', 'price': 199},
    {'name': 'USB-C Hub', 'price': 89},
    {'name': 'Laptop Stand', 'price': 79},
]
searcher = ProductSearch(ExactMatch())
print('Exact "mouse":', [p['name'] for p in searcher.search('mouse', products)])
searcher._strategy = FuzzyMatch()
print('Fuzzy "wireless keyboard":', [p['name'] for p in searcher.search('wireless keyboard', products)])

In [ ]:
# Cwiczenie 2: TaxStrategy z rejestrem
tax_registry = StrategyRegistry()

@tax_registry.register('PL')
def poland_tax(income: float) -> float:
    if income <= 120000: return income * 0.12
    return 14400 + (income - 120000) * 0.32

@tax_registry.register('DE')
def germany_tax(income: float) -> float:
    return income * 0.19

@tax_registry.register('UK')
def uk_tax(income: float) -> float:
    if income <= 12570: return 0
    if income <= 50270: return (income - 12570) * 0.20
    return 7540 + (income - 50270) * 0.40

for country, income in [('PL', 100000), ('DE', 100000), ('UK', 100000)]:
    tax_fn = tax_registry.create(country)
    tax = tax_fn(income)
    print(f'{country}: income={income:,}, tax={tax:,.2f} ({tax/income*100:.1f}%)')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: BackoffStrategy
import time

class BackoffStrategy(ABC):
    @abstractmethod
    def wait_time(self, attempt: int) -> float: ...

class ConstantBackoff(BackoffStrategy):
    def __init__(self, delay: float): self._delay = delay
    def wait_time(self, attempt: int) -> float: return self._delay

class ExponentialBackoff(BackoffStrategy):
    def __init__(self, base: float = 2.0, max_delay: float = 60.0):
        self._base = base; self._max = max_delay
    def wait_time(self, attempt: int) -> float:
        return min(self._base ** attempt, self._max)

class LinearBackoff(BackoffStrategy):
    def __init__(self, increment: float = 1.0): self._inc = increment
    def wait_time(self, attempt: int) -> float: return attempt * self._inc

def retry(func, strategy: BackoffStrategy, max_attempts: int = 5):
    import random
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except Exception as e:
            if attempt == max_attempts: raise
            wait = strategy.wait_time(attempt)
            print(f'Attempt {attempt} failed: {e}. Wait: {wait:.2f}s')
            # time.sleep(wait)  # nie czekamy w demonstracji

attempts = [0]
def flaky():
    attempts[0] += 1
    if attempts[0] < 4: raise ConnectionError('timeout')
    return 'success!'

for strategy_name, strategy in [('Constant', ConstantBackoff(1)), ('Exponential', ExponentialBackoff(2)), ('Linear', LinearBackoff(1))]:
    attempts[0] = 0
    print(f'\n{strategy_name}:')
    result = retry(flaky, strategy)
    print(f'Result: {result}')